In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload


In [ ]:
import sys
import numpy as np 
import pandas as pd
from pathlib import Path
from src.phaseI.main import run_phase1
from src.phaseII.main import run_phase2

from demo.core.data import (
    SplitConfig,
    prepare_cmapss_split,
)


config = SplitConfig(
    data_path="demo/data/train_FD001.txt",

    train_ratio=0.70,

    elbow_cycle=150,
    elbow_tolerance=5,

    extended_probability=0.30,
    extended_max_extra=50,

    min_cycles_before_failure=20,

    test_fraction_min=0.10,
    test_fraction_max=0.30,

    random_seed=42,
)
data = prepare_cmapss_split(config)

from src.preprocess_data.realtime import (
    simulate_realtime,
    RealtimeConfig,
)

realtime = simulate_realtime(
    engine_data=data["engine_data"],
    test_data=data["test_data"],
    observation_points=data["test_observation_points"],
    config=RealtimeConfig(
        update_probability=0.8,
        max_new_cycles=1,
        random_seed=42,
    ),
)


In [3]:
BIN_STRIDE = 10

model, stats, normalized_data, latent_data = run_phase1(
    train_data=data["train_data"],
    n_sensors=len(data["sensors"]),
    window_len=30,
    latent_dim=16,
    bin_stride=BIN_STRIDE,
)


epoch   0 | recon_loss=0.641268
epoch   1 | recon_loss=0.459071
epoch   2 | recon_loss=0.455278
epoch   3 | recon_loss=0.452914
epoch   4 | recon_loss=0.451605
epoch   5 | recon_loss=0.449937
epoch   6 | recon_loss=0.448773
epoch   7 | recon_loss=0.447074
epoch   8 | recon_loss=0.444165
epoch   9 | recon_loss=0.442259
epoch  10 | recon_loss=0.440754
epoch  11 | recon_loss=0.438643
epoch  12 | recon_loss=0.436481
epoch  13 | recon_loss=0.434676
epoch  14 | recon_loss=0.432115
epoch  15 | recon_loss=0.428120
epoch  16 | recon_loss=0.423622
epoch  17 | recon_loss=0.420929
epoch  18 | recon_loss=0.419783
epoch  19 | recon_loss=0.418983
epoch  20 | recon_loss=0.418373
epoch  21 | recon_loss=0.417992
epoch  22 | recon_loss=0.417828
epoch  23 | recon_loss=0.417501
epoch  24 | recon_loss=0.417374
epoch  25 | recon_loss=0.417133
epoch  26 | recon_loss=0.417036
epoch  27 | recon_loss=0.416860
epoch  28 | recon_loss=0.416732
epoch  29 | recon_loss=0.416614


In [5]:
from src.phaseII.core.prepare_data import build_engine_records
records = build_engine_records(
    latent_data=latent_data,
    train_metadata=data["train_metadata"],
)

In [6]:
print("\n=== ENGINE RECORDS ===")

for record in records:
    print(
        f"engine={record.engine_id:>3} | "
        f"shape={record.latent_seq.shape} | "
        f"n_bins={record.n_bins:>3} | "
        f"event_observed={record.event_observed} | "
        f"event_bin={record.event_bin}"
    )


=== ENGINE RECORDS ===
engine=  1 | shape=(11, 16) | n_bins= 11 | event_observed=False | event_bin=None
engine=  2 | shape=(10, 16) | n_bins= 10 | event_observed=False | event_bin=None
engine=  3 | shape=(10, 16) | n_bins= 10 | event_observed=False | event_bin=None
engine=  4 | shape=(12, 16) | n_bins= 12 | event_observed=True | event_bin=11
engine=  5 | shape=(15, 16) | n_bins= 15 | event_observed=True | event_bin=14
engine=  6 | shape=(10, 16) | n_bins= 10 | event_observed=False | event_bin=None
engine=  8 | shape=(11, 16) | n_bins= 11 | event_observed=False | event_bin=None
engine= 10 | shape=(11, 16) | n_bins= 11 | event_observed=False | event_bin=None
engine= 11 | shape=(10, 16) | n_bins= 10 | event_observed=False | event_bin=None
engine= 16 | shape=(15, 16) | n_bins= 15 | event_observed=True | event_bin=14
engine= 17 | shape=(11, 16) | n_bins= 11 | event_observed=True | event_bin=10
engine= 18 | shape=(10, 16) | n_bins= 10 | event_observed=False | event_bin=None
engine= 19 | sha